# AI Flight Booking Agent
```
User: 'Mumbai to Delhi tomorrow one way'
         |
      Whisper -> text
         |
      LLaMA -> extract {departure, destination, date, trip_type}
         |
      scraper.py -> Google Flights -> saves CSV to Drive
         |
      LLaMA reads CSV -> 'Found 12 flights! Want cheapest / preferred airline / preferred timing?'
         |
      User: 'cheapest'  ->  show top 5 sorted by price
      User: 'IndiGo'    ->  filter by that airline
      User: 'morning'   ->  filter by departure time
         |
      User picks 1-5  ->  LLaMA confirms booking
```
**One-time setup:** Upload `scraper.py` to `My Drive/flight_agent/scraper.py`

In [5]:
!pip install -q 'fast-flights @ git+https://github.com/AWeirdDev/flights.git@dev' --force-reinstall
!pip install -q selectolax pandas gradio gtts transformers accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 81.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.34.1 which is incompatible.
ydf 0

In [6]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.8 MB/s eta 0:00:00


In [4]:
!pip install -U transformers tenacity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 37.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [7]:
from google.colab import drive
import sys, os, pathlib

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/flight_agent'
CACHE_DIR  = os.path.join(DRIVE_DIR, 'flight_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

if DRIVE_DIR not in sys.path:
    sys.path.insert(0, DRIVE_DIR)

if not os.path.exists(os.path.join(DRIVE_DIR, 'scraper.py')):
    raise FileNotFoundError(
        'scraper.py not found! Upload it to My Drive/flight_agent/scraper.py'
    )

print('scraper.py loaded from Drive')
print(f'CSV cache folder: {CACHE_DIR}')


Mounted at /content/drive
scraper.py loaded from Drive
CSV cache folder: /content/drive/MyDrive/flight_agent/flight_cache


In [8]:
from google.colab import userdata
from huggingface_hub import login

# Sign in to HuggingFace Hub

hf_token = userdata.get('myHF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
import os
os.kill(os.getpid(), 9)

In [9]:

# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, pipeline
import gradio as gr
import torch

In [10]:
def load_models():
    LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )
    print("🔄 Loading LLaMA...")
    tokenizer = AutoTokenizer.from_pretrained(LLAMA)
    tokenizer.pad_token = tokenizer.eos_token
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLAMA, device_map="auto", quantization_config=quant_config
    )
    print("✅ LLaMA ready!")

    print("🔄 Loading Whisper...")
    asr_pipe = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-medium.en",
        dtype=torch.float16, device=0,
        return_timestamps=True
    )
    print("✅ Whisper ready!")
    return tokenizer, llm_model, asr_pipe

tokenizer, llm_model, asr_pipe = load_models()

🔄 Loading LLaMA...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


✅ Whisper ready!


In [25]:
import re, json, os, tempfile, pathlib
import pandas as pd
from datetime import datetime, timedelta
from gtts import gTTS
import soundfile as sf
import librosa
import numpy as np
CACHE = pathlib.Path(CACHE_DIR)
import scraper

# ─────────────────────────────────────────
# LOAD AIRPORT DATA (for robust matching)
# ─────────────────────────────────────────
airports_df = pd.read_csv(os.path.join(DRIVE_DIR, 'airports.csv'))


# ─────────────────────────────────────────
# INTENT EXTRACTION (FIXED)
# ─────────────────────────────────────────
def extract_route_fast(text):
    text = text.lower()

    # remove filler words safely
    text = re.sub(r'\b(i want to|i want|book|flight ticket|ticket|please|can you|me)\b', '', text)
    text = re.sub(r'\b(from)\b', '', text)

    # remove noise
    text = re.sub(r'\b(today|tomorrow|one way|one-way|round trip|round-trip)\b', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    match = re.search(r'([a-zA-Z ]+?)\s+to\s+([a-zA-Z ]+)', text)
    if not match:
        return None

    dep = match.group(1).strip().title()
    dst = match.group(2).strip().title()

    dst = re.split(r'\b(flight|ticket)\b', dst)[0].strip()

    return {"departure": dep, "destination": dst}


def extract_route_with_airports(text):
    text = text.lower()
    cities = airports_df['city'].dropna().unique()

    found = [c for c in cities if c.lower() in text]

    if len(found) >= 2:
        return {
            "departure": found[0].title(),
            "destination": found[1].title()
        }
    return None

def find_city_in_text(text):
    text = text.lower()
    cities = airports_df['city'].dropna().unique()

    found = []
    for city in cities:
        if city.lower() in text:
            found.append(city)

    return found

def adjust_date_if_needed(date_str, user_text=""):
    if not date_str:
        return None, False

    try:
        dt = datetime.strptime(date_str, "%Y-%m-%d")

        if "tomorrow" in user_text.lower() or "today" in user_text.lower():
            return date_str, False

        if dt < datetime.now() + timedelta(days=1):
            new_dt = datetime.now() + timedelta(days=3)
            return new_dt.strftime("%Y-%m-%d"), True
    except:
        pass

    return date_str, False

def llm_extract_intent(user_text):
    fast = extract_route_fast(user_text)

    if not fast:
        fast = extract_route_with_airports(user_text)

    today = datetime.now()
    date = None

    if "tomorrow" in user_text.lower():
        date = (today + timedelta(days=1)).strftime('%Y-%m-%d')
    elif "today" in user_text.lower():
        date = today.strftime('%Y-%m-%d')

    if fast:
        new_date, adjusted = adjust_date_if_needed(date, user_text)

        return {
            "departure": fast["departure"],
            "destination": fast["destination"],
            "date": new_date,
            "trip_type": "one-way",
            "date_adjusted": adjusted
        }

    return {'departure':None,'destination':None,'date':None,'trip_type':'one-way'}


# ─────────────────────────────────────────
# SCRAPER
# ─────────────────────────────────────────
def llm_call(prompt, max_tokens=200):
    inputs = tokenizer(prompt, return_tensors='pt').to(llm_model.device)

    out = llm_model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(out[0], skip_special_tokens=True).split('Assistant:')[-1].strip()


def cache_path(dep, dst, date):
    dep = dep.lower().replace(' ','_')
    dst = dst.lower().replace(' ','_')
    return CACHE / f'{dep}_to_{dst}_{date}.csv'


import os

def run_scraper(dep, dst, date, trip):
    path = cache_path(dep, dst, date)

    # ✅ check cache
    if path.exists():
        df = pd.read_csv(path)
        print(f'Loaded cache: {path.name}')
        return df

    print(f"Scraping: {dep} -> {dst} on {date}")

    # ✅ call scraper
    flights = scraper.scrape(dep, dst, date, trip=trip)

    df = pd.DataFrame(flights)

    # ✅ save cache
    df.to_csv(path, index=False)
    print(f"Saved cache: {path.name}")

    return df


# ─────────────────────────────────────────
# LLM RESPONSE (CLEAN PROMPT)
# ─────────────────────────────────────────
def llm_describe_flights(df, dep, dst, date):
    total = len(df)
    cheapest = df.loc[df['price'].idxmin()]
    fastest = df.loc[df['duration_min'].idxmin()]
    min_price = df['price'].min()
    max_price = df['price'].max()

    return (
        f"I found {total} flights from {dep} to {dst} on {date}. "
        f"Prices range from Rs {min_price} to Rs {max_price}. "
        f"The cheapest is {cheapest['airline']} at Rs {cheapest['price']}. "
        f"The fastest takes {fastest['duration']}. "
        f"Do you prefer cheapest, fastest, non-stop, or a specific airline?"
    )


# ─────────────────────────────────────────
# FILTER
# ─────────────────────────────────────────
def apply_preference(df, user_text):
    """
    Parse user preference from their message and filter/sort df.
    Returns (filtered_df, description_of_what_was_applied)
    """
    u = user_text.lower()
    result = df.copy()
    applied = []

    # ── Non-stop filter ───────────────────────────────────────
    if any(w in u for w in ['non-stop','nonstop','non stop','direct']):
        ns = result[result['stops_raw'] == 0]
        if not ns.empty:
            result = ns
            applied.append('non-stop only')

    # ── Time of day filter ────────────────────────────────────
    if any(w in u for w in ['morning','early']):
        result['_dep_h'] = result['departure_str'].apply(
            lambda x: int(str(x)[11:13]) if len(str(x)) >= 13 else 12
        )
        t = result[result['_dep_h'] < 12]
        if not t.empty:
            result = t
            applied.append('morning departures (before 12 PM)')
        result = result.drop(columns=['_dep_h'], errors='ignore')

    elif any(w in u for w in ['afternoon']):
        result['_dep_h'] = result['departure_str'].apply(
            lambda x: int(str(x)[11:13]) if len(str(x)) >= 13 else 12
        )
        t = result[(result['_dep_h'] >= 12) & (result['_dep_h'] < 17)]
        if not t.empty:
            result = t
            applied.append('afternoon departures (12-5 PM)')
        result = result.drop(columns=['_dep_h'], errors='ignore')

    elif any(w in u for w in ['evening','night']):
        result['_dep_h'] = result['departure_str'].apply(
            lambda x: int(str(x)[11:13]) if len(str(x)) >= 13 else 12
        )
        t = result[result['_dep_h'] >= 17]
        if not t.empty:
            result = t
            applied.append('evening departures (after 5 PM)')
        result = result.drop(columns=['_dep_h'], errors='ignore')

    # ── Airline filter ────────────────────────────────────────
    known_airlines = [
        'indigo','air india','spicejet','vistara','go first',
        'akasa','emirates','etihad','qatar','singapore airlines',
        'british airways','lufthansa','klm','air france',
    ]
    for airline in known_airlines:
        if airline in u:
            mask = result['airline'].str.lower().str.contains(airline, regex=False)
            t    = result[mask]
            if not t.empty:
                result = t
                applied.append(f'{airline.title()} flights')
            break

    # ── Sort ──────────────────────────────────────────────────
    if any(w in u for w in ['cheap','cheapest','lowest price','price','budget']):
        result = result.sort_values('price')
        applied.append('sorted by price')
    elif any(w in u for w in ['fast','fastest','shortest','quick','duration']):
        result = result.sort_values('duration_min')
        applied.append('sorted by duration')
    elif any(w in u for w in ['stop','stops','fewest stops']):
        result = result.sort_values('stops_raw')
        applied.append('sorted by stops')
    else:
        # default: cheapest
        result = result.sort_values('price')
        applied.append('sorted by price')

    desc = ', '.join(applied) if applied else 'sorted by price'
    return result, desc



def build_flight_table(df, label):
    """
    Build a numbered text table of top 5 flights.
    """
    top5 = df.head(5)
    lines = [f'Top {len(top5)} flights ({label}):\n']
    for i, (_, r) in enumerate(top5.iterrows(), 1):
        lines.append(
            f'{i}. {str(r["airline"]):<28} '
            f'{r["departure_str"]} -> {r["arrival_str"]}  '
            f'{r["duration"]:<8} {r["stops"]:<14} '
            f'{r["price_str"]}'
        )
    lines.append('\nReply with 1-5 to select, or ask for different filter.')
    lines.append('(e.g. "morning flights", "IndiGo only", "non-stop", "cheapest")')
    return '\n'.join(lines)


def llm_confirm(chosen):
    prompt = (
        'You are a friendly flight booking assistant. Reply in 2 sentences only.\n'
        f'Airline: {chosen["airline"]}\n'
        f'Departure: {chosen["departure_str"]}\n'
        f'Arrival: {chosen["arrival_str"]}\n'
        f'Price: {chosen["price_str"]}\n'
        f'Duration: {chosen["duration"]}\n'
        f'Stops: {chosen["stops"]}\n\n'
        'Confirm the booking warmly. Say support agent will contact for payment.\n'
        'Assistant:'
    )
    result = llm_call(prompt, max_tokens=100)
    if any(x in result for x in ['A)','B)','1.','2.','?','option']):
        return (
            f'Great choice! Your {chosen["airline"]} flight '
            f'departing {chosen["departure_str"]} '
            f'for {chosen["price_str"]} is confirmed. '
            'Our support agent will contact you for payment details. Safe travels!'
        )
    return result
# ─────────────────────────────────────────
# SESSION
# ─────────────────────────────────────────

def new_session():
    return {
        'stage':       'collect',
        # collect -> scraping -> preference -> pick -> done
        'departure':   None,
        'destination': None,
        'date':        None,
        'trip_type':   'one-way',
        'df':          None,   # full DataFrame from scraper
        'top5':        None,   # filtered top-5 rows
        'chosen':      None,
        'history':     [],
    }

def fmt_history(s):
    return '\n'.join(
        f'You: {h["user"]}\nAgent: {h["assistant"]}\n'
        for h in s['history'][-8:]
    )


def handle_turn(user_text, session):
    u = user_text.lower().strip()

    # ── Reset ─────────────────────────────────────────────────
    if any(w in u for w in ['reset','restart','new search','start over','new flight']):
        session.update(new_session())
        return 'Starting fresh! Tell me where and when you want to fly.', session

    # ══════════════════════════════════════════════════════════
    # STAGE: collect
    # LLaMA extracts departure/destination/date from user text
    # ══════════════════════════════════════════════════════════
    if session['stage'] == 'collect':
        print(f'Extracting intent: "{user_text}"')
        intent = llm_extract_intent(user_text)
        print(f'Intent: {intent}')

        if intent.get('departure'):   session['departure']   = intent['departure']
        if intent.get('destination'): session['destination'] = intent['destination']
        if intent.get('date'):        session['date']        = intent['date']
        if intent.get('trip_type'):   session['trip_type']   = intent['trip_type']

        missing = []
        if not session['departure']:   missing.append('departure city')
        if not session['destination']: missing.append('destination city')
        if not session['date']:        missing.append('travel date')

        if missing:
            return f'Got it! Could you also tell me your {", ".join(missing)}?', session

        # All collected -> go scrape
        session['stage'] = 'scraping'
        return handle_turn('__scrape__', session)

    # ══════════════════════════════════════════════════════════
    # STAGE: scraping
    # Call scraper.py -> save CSV to Drive -> LLaMA describes results
    # ══════════════════════════════════════════════════════════
    if session['stage'] == 'scraping':
        dep  = session['departure']
        dst  = session['destination']
        date = session['date']
        trip = session['trip_type']

        try:
            df = run_scraper(dep, dst, date, trip)

            if df.empty:
                session.update(new_session())
                return (
                    f'Sorry, no flights found from {dep} to {dst} on {date}. '
                    'Try a different date or route.',
                    session
                )

            session['df']    = df
            session['stage'] = 'preference'


            pretty_date = datetime.strptime(date, "%Y-%m-%d").strftime("%b %d")

            note = ""
            if session.get("date_adjusted"):
               note = (
                      f"Note: Flights for your selected date were limited, "
                      f"so I checked availability on {pretty_date} for better options.\n\n"
                   )

            response = llm_describe_flights(df, dep, dst, pretty_date)

            return note + response, session

        except Exception as e:
            session.update(new_session())
            return (
                f'Could not fetch flights from {dep} to {dst} on {date}. '
                f'Error: {str(e)[:100]}. Please try again.',
                session
            )

    # ══════════════════════════════════════════════════════════
    # STAGE: preference
    # User says cheapest / IndiGo / morning / non-stop etc.
    # Filter+sort CSV, show top 5
    # ══════════════════════════════════════════════════════════
    if session['stage'] == 'preference':
        df = session['df']
        filtered, label = apply_preference(df, user_text)

        if filtered.empty:
            return (
                'No flights match that filter. '
                'Try: cheapest / morning / evening / non-stop / airline name.',
                session
            )

        # Store top 5 as list of dicts for easy indexing
        session['top5']  = filtered.head(5).to_dict(orient='records')
        session['stage'] = 'pick'
        return build_flight_table(filtered, label), session

    # ══════════════════════════════════════════════════════════
    # STAGE: pick
    # User types 1-5 OR gives another preference filter
    # ══════════════════════════════════════════════════════════
    if session['stage'] == 'pick':
        # convert words → numbers
        word_to_num = {
                "one": "1",
                   "two": "2",
                     "three": "3",
                        "four": "4",
                        "five": "5"
         }

        for word, num in word_to_num.items():
          if word in u:
            u = u.replace(word, num)
        nums = re.findall(r'\b([1-5])\b', u)
        if nums:
            idx    = int(nums[0]) - 1
            chosen = session['top5'][idx]
            session['chosen'] = chosen
            session['stage']  = 'done'
            return llm_confirm(chosen), session

        # User gave another filter instead of picking
        # go back to preference stage with same df
        session['stage'] = 'preference'
        return handle_turn(user_text, session)

    # ══════════════════════════════════════════════════════════
    # STAGE: done
    # ══════════════════════════════════════════════════════════
    if session['stage'] == 'done':
        return (
            'Your booking is confirmed! Our agent will contact you for payment. '
            'Type "reset" to search for another flight.',
            session
        )

    return 'Something went wrong. Type "reset" to start over.', session



# ─────────────────────────────────────────
# AUDIO FIX (CRITICAL)
# ─────────────────────────────────────────
def transcribe(audio_path):
    if not audio_path:
        return ""

    try:
        audio, sr = librosa.load(audio_path, sr=16000)
        result = asr_pipe(audio)

        if isinstance(result, dict):
            return result.get("text", "").strip()

        return str(result)

    except Exception as e:
        print("ASR ERROR:", e)
        return ""


def tts(text):
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
    gTTS(text).save(tmp.name)
    return tmp.name


print("✅ FULL AGENT READY")

✅ FULL AGENT READY


In [22]:
import shutil

shutil.rmtree(CACHE_DIR, ignore_errors=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print("Cache cleared")

Cache cleared


In [17]:
def cache_path(dep, dst, date):
    dep = dep.lower().replace(' ', '_')
    dst = dst.lower().replace(' ', '_')
    return os.path.join(CACHE_DIR, f'{dep}_to_{dst}_{date}.csv')

In [26]:
import gradio as gr

def process(audio, text_input, session):
    user_text = ""

    print("AUDIO FILE:", audio)
    print("TEXT INPUT:", text_input)

    # 🎤 Priority 1: audio
    if audio:
        print("🎤 Audio received:", audio)
        try:
            user_text = transcribe(audio)
            print("TRANSCRIBED:", user_text)
        except Exception as e:
            print("❌ ASR ERROR:", e)
            user_text = ""

    # ⌨️ Priority 2: fallback to text
    if not user_text:
        user_text = (text_input or "").strip()

    # ❗ still empty
    if not user_text:
        return "", "Please type or speak your travel details (e.g. Goa to Bangalore tomorrow)", None, "", session

    # 🚀 main agent
    response, session = handle_turn(user_text, session)

    # 🧠 store history safely
    if 'history' not in session:
        session['history'] = []

    session['history'].append({
        'user': user_text,
        'assistant': response
    })

    # 🔊 generate voice safely
    try:
        audio_reply = tts(response)
    except Exception as e:
        print("TTS ERROR:", e)
        audio_reply = None

    return user_text, response, audio_reply, fmt_history(session), session
with gr.Blocks(title='AI Flight Booking Agent') as demo:
    gr.Markdown(
        """
        <div style="text-align:center;">
            <h2>✈️ AI Flight Booking Agent</h2>
            <p><b>Project by Gayathri VR</b></p>
        </div>

        **How to use:**
        1. Say or type: *"Goa to Bangalore tomorrow one way"*
        2. Agent scrapes Google Flights and tells you what it found
        3. Say your preference: *"cheapest"* / *"morning flights"* / *"IndiGo"* / *"non-stop"*
        4. Pick a flight: *"2"*
        5. Done! Type *"reset"* to search again
        """
    )
    state = gr.State(new_session())

    with gr.Row():
        with gr.Column(scale=1):
            audio_in  = gr.Audio(
                sources=['microphone'], type='filepath', label='Speak'
            )
            text_in   = gr.Textbox(
                label='Or type here',
                placeholder='e.g. Goa to Bangalore tomorrow one way'
            )
            with gr.Row():
                send_btn  = gr.Button('Send',  variant='primary')
                reset_btn = gr.Button('Reset', variant='secondary')

            gr.Markdown(
                '**Preference examples after results:**\n'
                '`cheapest` `morning` `afternoon` `evening`\n'
                '`non-stop` `IndiGo` `Air India` `SpiceJet`\n'
                '`fastest` `fewest stops`'
            )

        with gr.Column(scale=1):
            said_box  = gr.Textbox(label='You said',    interactive=False)
            reply_box = gr.Textbox(label='Agent reply', interactive=False, lines=14)
            audio_out = gr.Audio( label='Voice reply',  autoplay=True)

    history_box = gr.Textbox(
        label='Conversation history', lines=10, interactive=False
    )

    send_btn.click(
        fn=process,
        inputs=[audio_in, text_in, state],
        outputs=[said_box, reply_box, audio_out, history_box, state]
    )
    text_in.submit(
        fn=lambda t, s: process(None, t, s),
        inputs=[text_in, state],
        outputs=[said_box, reply_box, audio_out, history_box, state]
    )
    reset_btn.click(
        fn=lambda: ('','Tell me where and when you want to fly!',None,'',new_session()),
        outputs=[said_box, reply_box, audio_out, history_box, state]
    )

demo.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://323ee48a70c1780a0c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


AUDIO FILE: /tmp/gradio/4d0e14149a73c645b324edc62fc5a21b4be18594fcaa413a7c75ffb4035dcb3c/audio.wav
TEXT INPUT: 
🎤 Audio received: /tmp/gradio/4d0e14149a73c645b324edc62fc5a21b4be18594fcaa413a7c75ffb4035dcb3c/audio.wav
TRANSCRIBED: hi I want to book flight ticket from Delhi to Mumbai tomorrow one way
Extracting intent: "hi I want to book flight ticket from Delhi to Mumbai tomorrow one way"
Intent: {'departure': 'Hi Delhi', 'destination': 'Mumbai', 'date': '2026-05-08', 'trip_type': 'one-way', 'date_adjusted': False}
Scraping: Hi Delhi -> Mumbai on 2026-05-08

Scraping: Hi Delhi (HI ) -> Mumbai (BOM)
Date: 2026-05-08  Trip: one-way  Seat: economy  Adults: 1
URL: https://www.google.com/travel/flights/search?tfs=GhoSCjIwMjYtMDUtMDhqBRIDSEkgcgUSA0JPTUIBAUgBmAEC&hl=en-US&curr=

Attempt 1/3...
[]
  Error: list index out of range
  Retrying...
Attempt 2/3...
[]
  Error: list index out of range
  Retrying...
Attempt 3/3...
[]
  Error: list index out of range


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


AUDIO FILE: /tmp/gradio/3a66c17722bb68aa3252bf521125e14f4f9b4120cf818e383f2e4b20443fe767/audio.wav
TEXT INPUT: 
🎤 Audio received: /tmp/gradio/3a66c17722bb68aa3252bf521125e14f4f9b4120cf818e383f2e4b20443fe767/audio.wav
TRANSCRIBED: I want to book flight ticket from Mumbai to Delhi tomorrow one day.
Extracting intent: "I want to book flight ticket from Mumbai to Delhi tomorrow one day."
Intent: {'departure': 'Mumbai', 'destination': 'Delhi One Day', 'date': '2026-05-08', 'trip_type': 'one-way', 'date_adjusted': False}
Scraping: Mumbai -> Delhi One Day on 2026-05-08

Scraping: Mumbai (BOM) -> Delhi One Day (DEL)
Date: 2026-05-08  Trip: one-way  Seat: economy  Adults: 1
URL: https://www.google.com/travel/flights/search?tfs=GhoSCjIwMjYtMDUtMDhqBRIDQk9NcgUSA0RFTEIBAUgBmAEC&hl=en-US&curr=

Attempt 1/3...
[[null,[[1778162193203812,16325341,892071020],null,null,null,null,[[1]]],0,"EZr8aaS4DN215LcP7NivqQM","HC07XYhVUzaAAC_NfgBG--------pfbkq15AAAAAGn8mhEDGRRSA"],[[[[["BOM",0],"Chhatrapati Shivaji 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


TRANSCRIBED: I'll go with the five
AUDIO FILE: None
TEXT INPUT: 
AUDIO FILE: /tmp/gradio/3bfb0288e8270e950b4fa25b0bf8798b61a24be5230bd9d5d233a5f6f9912681/audio.wav
TEXT INPUT: 
🎤 Audio received: /tmp/gradio/3bfb0288e8270e950b4fa25b0bf8798b61a24be5230bd9d5d233a5f6f9912681/audio.wav
TRANSCRIBED: I want to book fly ticket from Bangalore to Goa on next month 8
Extracting intent: "I want to book fly ticket from Bangalore to Goa on next month 8"
Intent: {'departure': 'Fly Bangalore', 'destination': 'Goa On Next Month', 'date': None, 'trip_type': 'one-way', 'date_adjusted': False}
AUDIO FILE: None
TEXT INPUT: 
AUDIO FILE: /tmp/gradio/23ade3d47124047b5f38aeebeddc406210b3f3ae654b5ff0e111cb9e6c326e60/audio.wav
TEXT INPUT: 
🎤 Audio received: /tmp/gradio/23ade3d47124047b5f38aeebeddc406210b3f3ae654b5ff0e111cb9e6c326e60/audio.wav
TRANSCRIBED: I want to go to Goa to Bangalore tomorrow round trip
Extracting intent: "I want to go to Goa to Bangalore tomorrow round trip"
Intent: {'departure': 'Go', 'des